# Global Solution 2026 - NeuroSpace Alert

Simulador simples de um sensor neuromórfico de baixo consumo para detecção de condição crítica em ambiente espacial ou em estação remota monitorada por satélite.

## 1. Upload do dataset

Execute a célula abaixo e envie o arquivo `dataset_neurosensor_espacial_5h.csv`.

In [1]:
import pandas as pd
import numpy as np

try:
    from google.colab import files
    uploaded = files.upload()
    nome_arquivo = list(uploaded.keys())[0]
except Exception:
    # Caso esteja rodando fora do Colab, coloque o CSV na mesma pasta do notebook.
    nome_arquivo = 'dataset_neurosensor_espacial_5h.csv'

df = pd.read_csv(nome_arquivo)
print('Dataset carregado com sucesso!')
print('Linhas e colunas:', df.shape)
df.head()


Saving dataset_neurosensor_espacial_5h.csv to dataset_neurosensor_espacial_5h.csv
Dataset carregado com sucesso!
Linhas e colunas: (61, 8)


,tempo_min,temperatura_C,radiacao_uSv_h,poeira_pct,consumo_W,bateria_pct,indice_risco_real,condicao_real
0,0,22.6,0.27,11.5,18.5,99.1,0.060,NORMAL
1,5,22.2,0.23,6.0,18.0,99.1,0.040,NORMAL
2,10,22.8,0.23,10.1,18.7,98.4,0.056,NORMAL
3,15,23.5,0.32,13.5,18.6,97.6,0.083,NORMAL
4,20,22.3,0.35,7.3,18.2,98.7,0.060,NORMAL


## 2. Conferência rápida dos dados

O dataset simula 5 horas de operação, com uma leitura a cada 5 minutos.

In [2]:
print('Primeiras linhas:')
display(df.head())

print('\nÚltimas linhas:')
display(df.tail())

print('\nResumo da condição real simulada:')
display(df['condicao_real'].value_counts())


Primeiras linhas:


,tempo_min,temperatura_C,radiacao_uSv_h,poeira_pct,consumo_W,bateria_pct,indice_risco_real,condicao_real
0,0,22.6,0.27,11.5,18.5,99.1,0.060,NORMAL
1,5,22.2,0.23,6.0,18.0,99.1,0.040,NORMAL
2,10,22.8,0.23,10.1,18.7,98.4,0.056,NORMAL
3,15,23.5,0.32,13.5,18.6,97.6,0.083,NORMAL
4,20,22.3,0.35,7.3,18.2,98.7,0.060,NORMAL



Últimas linhas:


,tempo_min,temperatura_C,radiacao_uSv_h,poeira_pct,consumo_W,bateria_pct,indice_risco_real,condicao_real
56,280,48.5,2.43,75.3,30.3,60.0,0.856,CRITICA
57,285,49.0,2.53,81.5,29.0,59.6,0.886,CRITICA
58,290,49.6,2.53,77.6,29.6,59.9,0.886,CRITICA
59,295,50.1,2.53,74.8,30.2,59.2,0.887,CRITICA
60,300,49.2,2.47,74.6,30.5,58.1,0.870,CRITICA



Resumo da condição real simulada:


,count
condicao_real,
NORMAL,31
CRITICA,21
ATENCAO,9


## 3. Função do sensor neuromórfico

A função abaixo converte temperatura, radiação e poeira em uma tensão de entrada. Depois, simula um memristor virtual com memória local.

In [3]:
def simular_sensor(dados, V_limiar, taxa_chaveamento,
                   limiar_estado_alerta=0.65,
                   limiar_estado_observacao=0.35):
    saida = dados.copy()

    # Normalização simples das variáveis físicas
    temp_norm = np.clip((saida['temperatura_C'] - 20) / (55 - 20), 0, 1)
    rad_norm = np.clip((saida['radiacao_uSv_h'] - 0.25) / (2.5 - 0.25), 0, 1)
    poeira_norm = np.clip(saida['poeira_pct'] / 100, 0, 1)

    # Conversão conceitual para tensão de entrada do circuito
    indice_sensor = 0.45 * temp_norm + 0.35 * rad_norm + 0.20 * poeira_norm
    saida['V_entrada'] = np.round(15 + 25 * indice_sensor, 2)

    estado_memristor = []
    led = []
    diagnostico_sensor = []
    w = 0.0
    dt = 5  # intervalo entre leituras, em minutos

    for v in saida['V_entrada']:
        if v > V_limiar:
            # O memristor acumula efeito quando a tensão passa do limiar
            w = min(1.0, w + taxa_chaveamento * (v - V_limiar) * dt * (1 - w))
        else:
            # Pequeno relaxamento quando o sinal fica abaixo do limiar
            w = max(0.0, w - 0.002 * dt * w)

        estado_memristor.append(round(w, 3))

        if w >= limiar_estado_alerta:
            led.append('VERMELHO')
            diagnostico_sensor.append('ALERTA_CRITICO')
        elif w >= limiar_estado_observacao:
            led.append('AMARELO')
            diagnostico_sensor.append('OBSERVACAO')
        else:
            led.append('APAGADO')
            diagnostico_sensor.append('NORMAL')

    saida['estado_memristor'] = estado_memristor
    saida['LED'] = led
    saida['diagnostico_sensor'] = diagnostico_sensor
    return saida


## 4. Ajustes do grupo

Altere pelo menos um dos três ajustes abaixo, mantendo os valores dentro das faixas indicadas no enunciado.

In [8]:
ajustes_da_equipe = {
    'Ajuste_A_sensivel': {'V_limiar': 24, 'taxa_chaveamento': 0.006},
    'Ajuste_B_equilibrado': {'V_limiar': 28, 'taxa_chaveamento': 0.006},
    'Ajuste_C_conservador': {'V_limiar': 32, 'taxa_chaveamento': 0.004},
}

ajustes_da_equipe

{'Ajuste_A_sensivel': {'V_limiar': 24, 'taxa_chaveamento': 0.006},
 'Ajuste_B_equilibrado': {'V_limiar': 28, 'taxa_chaveamento': 0.006},
 'Ajuste_C_conservador': {'V_limiar': 32, 'taxa_chaveamento': 0.004}}

## 5. Rodar a simulação

A célula abaixo executa os três ajustes e mostra quando o sensor começou a indicar observação e alerta crítico.

In [12]:
resultados = {}
resumo = []

for nome, pars in ajustes_da_equipe.items():
    saida = simular_sensor(df, pars['V_limiar'], pars['taxa_chaveamento'])
    resultados[nome] = saida

    primeiro_nao_apagado = saida[saida['LED'] != 'APAGADO'].head(1)
    primeiro_vermelho = saida[saida['LED'] == 'VERMELHO'].head(1)

    resumo.append({
        'ajuste': nome,
        'V_limiar': pars['V_limiar'],
        'taxa_chaveamento': pars['taxa_chaveamento'],
        'primeiro_LED_nao_apagado_min': None if primeiro_nao_apagado.empty else int(primeiro_nao_apagado['tempo_min'].iloc[0]),
        'primeiro_LED_vermelho_min': None if primeiro_vermelho.empty else int(primeiro_vermelho['tempo_min'].iloc[0]),
    })

resumo_transicoes = pd.DataFrame(resumo)
display(resumo_transicoes)


,ajuste,V_limiar,taxa_chaveamento,primeiro_LED_nao_apagado_min,primeiro_LED_vermelho_min
0,Ajuste_A_sensivel,24,0.006,170,190
1,Ajuste_B_equilibrado,28,0.006,200,220
2,Ajuste_C_conservador,32,0.004,245,280


## 6. Ver a tabela de saída de um ajuste

Escolha um ajuste e observe as colunas calculadas pelo sensor.

In [10]:
ajuste_escolhido = 'Ajuste_B_equilibrado'  # troque aqui, se quiser
saida_escolhida = resultados[ajuste_escolhido]

colunas_para_ver = [
    'tempo_min', 'temperatura_C', 'radiacao_uSv_h', 'poeira_pct',
    'condicao_real', 'V_entrada', 'estado_memristor', 'LED', 'diagnostico_sensor'
]

display(saida_escolhida[colunas_para_ver].tail(20))


,tempo_min,temperatura_C,radiacao_uSv_h,poeira_pct,condicao_real,V_entrada,estado_memristor,LED,diagnostico_sensor
41,205,44.1,2.04,61.2,CRITICA,32.77,0.479,AMARELO,OBSERVACAO
42,210,44.5,2.07,66.4,CRITICA,33.27,0.561,AMARELO,OBSERVACAO
43,215,45.0,2.15,66.8,CRITICA,33.76,0.637,AMARELO,OBSERVACAO
44,220,44.7,2.22,69.0,CRITICA,34.05,0.703,VERMELHO,ALERTA_CRITICO
45,225,45.7,2.31,72.3,CRITICA,34.89,0.764,VERMELHO,ALERTA_CRITICO
46,230,46.3,2.28,69.2,CRITICA,34.81,0.812,VERMELHO,ALERTA_CRITICO
47,235,47.8,2.32,69.1,CRITICA,35.44,0.854,VERMELHO,ALERTA_CRITICO
48,240,47.7,2.33,69.8,CRITICA,35.48,0.887,VERMELHO,ALERTA_CRITICO
49,245,46.5,2.29,70.7,CRITICA,34.99,0.911,VERMELHO,ALERTA_CRITICO
50,250,48.2,2.39,72.9,CRITICA,36.03,0.932,VERMELHO,ALERTA_CRITICO


## 7. Gerar arquivo de saída

Execute a célula abaixo para salvar a tabela do ajuste escolhido. Esse arquivo pode ser anexado ao relatório.

In [11]:
nome_saida = 'saida_sensor_espacial_' + ajuste_escolhido + '.csv'
saida_escolhida.to_csv(nome_saida, index=False)
print('Arquivo gerado:', nome_saida)

try:
    from google.colab import files
    files.download(nome_saida)
except Exception:
    pass


Arquivo gerado: saida_sensor_espacial_Ajuste_B_equilibrado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Perguntas para responder no relatório

1. Qual cenário espacial ou de integração espaço-Terra o grupo escolheu?
2. Qual condição crítica o sensor está tentando detectar?
3. Quais valores finais foram usados nos três ajustes?
4. Em que tempo o LED deixou de ficar apagado em cada ajuste?
5. Em que tempo o LED ficou vermelho em cada ajuste?
6. Qual ajuste detectou primeiro? Ele parece confiável ou sensível demais?
7. Qual ajuste o grupo recomenda como protótipo conceitual e por quê?
8. Por que esse sensor pode ser considerado de baixo consumo e com processamento local?